# 05 COF 性质预测：从 baseline 到可信验证

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wanteen/COF-ML-Tutorial/blob/main/notebooks/05_cof_property_prediction.ipynb)

## Learning objectives
完成 preprocessing → baseline → random split → family-aware split → error interpretation。`CO2_uptake_demo` 是人工教学标签，**不能用于科研结论**。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
url='https://raw.githubusercontent.com/Wanteen/COF-ML-Tutorial/main/data/cof_demo.csv'
df=pd.read_csv(url)
df.head()

## 1. 为什么用 Pipeline？
预处理必须只从 training data 学习。把 scaler/encoder 和 model 放进同一个 Pipeline，可以减少在 split 前处理全部数据造成的 leakage。

In [ ]:
target='CO2_uptake_demo'
features=['family','functional_group','pore_A','void_fraction','density','N_fraction','O_fraction']
X=df[features]; y=df[target]
cat=['family','functional_group']; num=[c for c in features if c not in cat]
pre=ColumnTransformer([('num',StandardScaler(),num),('cat',OneHotEncoder(handle_unknown='ignore'),cat)])
model=Pipeline([('pre',pre),('rf',RandomForestRegressor(n_estimators=500,random_state=42,n_jobs=-1))])

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=42)
model.fit(X_train,y_train); pred=model.predict(X_test)
random_metrics={
 'MAE':mean_absolute_error(y_test,pred),
 'RMSE':mean_squared_error(y_test,pred)**0.5,
 'R2':r2_score(y_test,pred)}
print('Random split:',random_metrics)

In [ ]:
splitter=GroupShuffleSplit(n_splits=1,test_size=0.25,random_state=42)
train_idx,test_idx=next(splitter.split(X,y,groups=df['family']))
X_train2,X_test2=X.iloc[train_idx],X.iloc[test_idx]
y_train2,y_test2=y.iloc[train_idx],y.iloc[test_idx]
model.fit(X_train2,y_train2); pred2=model.predict(X_test2)
group_metrics={
 'MAE':mean_absolute_error(y_test2,pred2),
 'RMSE':mean_squared_error(y_test2,pred2)**0.5,
 'R2':r2_score(y_test2,pred2)}
print('Train families:',sorted(df.iloc[train_idx]['family'].unique()))
print('Test families :',sorted(df.iloc[test_idx]['family'].unique()))
print('Family-aware:',group_metrics)

## 2. Interpolation vs extrapolation
如果 random split 明显优于 family-aware split，说明模型更擅长在已见化学空间附近插值。对真正 screening 新 COF，这个差距本身就是重要结果。

验证方案必须匹配科研问题：如果未来目标是预测新的 linker/linkage/topology，就应该设计相应的 leave-family-out 或 scaffold-aware 测试。

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(y_test2,pred2,alpha=0.8)
lo=min(y_test2.min(),pred2.min()); hi=max(y_test2.max(),pred2.max())
plt.plot([lo,hi],[lo,hi],'--')
plt.xlabel('True'); plt.ylabel('Predicted'); plt.title('Family-aware test'); plt.show()

## Exercises
1. 删除 functional group，比较性能。
2. 删除 pore features，比较性能。
3. 比较 random 与 family-aware 的 MAE/RMSE/R²。
4. 设计 leave-linker-out、leave-topology-out 或 publication-aware split。
5. 写一个 150 字的“模型适用域”说明，而不是只汇报最高 R²。

### Take-home message
材料 screening 模型的核心问题不是“测试集分数多高”，而是“测试集是否代表未来真正会遇到的未知材料”。